In [23]:
from dotenv import load_dotenv
import os

load_dotenv()

os.getenv("GEMINI_API_KEY")[:8]

'AIzaSyCs'

In [25]:
from google import genai
client = genai.Client()
print("Gemini OK")

Gemini OK


In [26]:
# store = client.file_search_stores.create(
#     config={"display_name": "kcw-kb-store"}
# )

# print("STORE:", store.name)

In [27]:
stores = list(client.file_search_stores.list())

print("Total stores:", len(stores))

for s in stores:
    print(s.name, "|", getattr(s, "display_name", None))

Total stores: 1
fileSearchStores/kcwkbstore-y3a6az4kie04 | kcw-kb-store


In [30]:
store = stores[0]

print("Using store:", store.name)

Using store: fileSearchStores/kcwkbstore-y3a6az4kie04


In [33]:
import re
from pathlib import Path

def clean_filename(path: str) -> str:
    name = Path(path).stem

    # remove non-ascii
    name = name.encode("ascii", "ignore").decode()

    # replace bad chars with _
    name = re.sub(r"[^A-Za-z0-9._-]+", "_", name)

    # collapse multiple _
    name = re.sub(r"_+", "_", name)

    # trim
    name = name.strip("_")

    # keep short
    return name[:60] or "doc"

In [37]:
from pathlib import Path
import time
import json

KB_FOLDER = Path(r"G:\Shared drives\KCW-Data\kcw_analytics\03_curated\knowledge_based")
REGISTRY_FILE = KB_FOLDER / "_uploaded_registry.json"

# ---------- load registry ----------
if REGISTRY_FILE.exists():
    uploaded = set(json.loads(REGISTRY_FILE.read_text()))
else:
    uploaded = set()

print("Already uploaded:", len(uploaded))

# ---------- upload loop ----------
for file_path in KB_FOLDER.glob("*.md"):

    name = file_path.name

    if name in uploaded:
        print("SKIP:", name)
        continue

    print("Uploading:", name)

    op = client.file_search_stores.upload_to_file_search_store(
        file=str(file_path),
        file_search_store_name=store.name,
        config={
            "display_name": name,
            "mime_type": "text/markdown"
        }
    )

    # wait indexing
    while not op.done:
        time.sleep(5)
        op = client.operations.get(op)

    print("Indexed:", name)

    uploaded.add(name)
    REGISTRY_FILE.write_text(json.dumps(list(uploaded), indent=2))

print("✅ ALL DONE")

Already uploaded: 0
Uploading: Vehicle_Parts_and_Bearings_Ref.md
Indexed: Vehicle_Parts_and_Bearings_Ref.md
Uploading: Aisin_Clutch_Technical_Master.md
Indexed: Aisin_Clutch_Technical_Master.md
Uploading: Kubota_Tractor_Full_Specs.md
Indexed: Kubota_Tractor_Full_Specs.md
Uploading: parts_measurement.md
Indexed: parts_measurement.md
Uploading: kubota_B.md
Indexed: kubota_B.md
Uploading: kubota_MU5501.md
Indexed: kubota_MU5501.md
Uploading: kubota_M8540_M9540.md
Indexed: kubota_M8540_M9540.md
Uploading: kubota_M7040.md
Indexed: kubota_M7040.md
Uploading: kubota_M6040.md
Indexed: kubota_M6040.md
Uploading: kubota_M108.md
Indexed: kubota_M108.md
Uploading: kubota_L5018.md
Indexed: kubota_L5018.md
Uploading: kubota_L3218_L4018.md
Indexed: kubota_L3218_L4018.md
Uploading: kubota_L3208_L3408_L3608.md
Indexed: kubota_L3208_L3408_L3608.md
Uploading: new_holland_6610S.md
Indexed: new_holland_6610S.md
Uploading: new_holland_TD95D.md
Indexed: new_holland_TD95D.md
Uploading: new_holland_TD90.md
Ind

In [39]:
FULL_SYSTEM_PROMPT = """
บทบาท:
คุณคือผู้เชี่ยวชาญด้านเทคนิคอะไหล่ยนต์และเครื่องจักร ตอบโดยอ้างอิงข้อมูลจากเอกสารในคลัง File Search เท่านั้น

รูปแบบการตอบ (LINE):
- ตอบภาษาไทย สั้น กระชับ อ่านง่าย
- บรรทัดแรก: สรุปคำตอบตรงๆ ไม่เกิน 2 ประโยค
- จากนั้นสรุปเป็นหัวข้อสั้นๆ 2–4 ข้อ
- หากมีข้อควรระวัง ให้ใส่ท้ายคำตอบแบบสั้น

กฎสำคัญ:
- ยึดข้อมูลจากเอกสารเป็นหลัก สามารถเชื่อมโยงคำที่มีความหมายใกล้เคียงได้ แต่ห้ามสร้างข้อมูลทางเทคนิคใหม่
- หากข้อมูลไม่ชัดเจน ให้ตอบว่า "ข้อมูลในคลังไม่ชัดเจน"
- หากไม่พบข้อมูล ให้ตอบว่า "ไม่มีข้อมูลในคลังข้อมูล"

การแสดงรูปภาพ:
- หากพบ Markdown รูปภาพในเอกสาร (เช่น ![alt](url)) ต้องคัดลอกมาแสดงแบบเดิมทันที ห้ามแก้ไข ห้ามย่อ ห้ามละเว้น

คำถามทั่วไป:
- หากไม่เกี่ยวกับงานหรืออะไหล่ ให้ตอบเชิงติดตลกว่าเป็นเวลางาน ให้ตั้งใจทำงานก่อน
"""

In [41]:
from google.genai import types

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="คำถามผู้ใช้: ซีลไทตัน",
    config=types.GenerateContentConfig(
        system_instruction=FULL_SYSTEM_PROMPT,
        tools=[
            types.Tool(
                file_search=types.FileSearch(
                    file_search_store_names=[store.name]
                )
            )
        ]
    )
)

print(response.text)

"ซีลไททัน" โดยตรงไม่พบในคลังข้อมูล แต่มี "ชุดซีลหัวฉีดไทรทัน" ซึ่งเป็นชุดอะไหล่สำหรับรถยนต์ Mitsubishi Triton รุ่น 4D56

รายละเอียดของชุดซีลหัวฉีดไทรทัน:
*   ยางปลอกหัวฉีด บน-เล็ก 2500-16V: ใช้ 4 ตัว
*   ประเก็นใต้ฝาวาว M/S TITAN 4D56: ใช้ 3 แผ่น
*   ยางปลอกหัวฉีดใหญ่ M/S TITAN 4D56T/16V: ใช้ 4 ตัว

ข้อควรระวัง: ชื่อ "ไททัน" ยังเกี่ยวข้องกับอะไหล่อื่นๆ เช่น ชุดสายพานไททัน และลูกปืนคลัชสำหรับค.ไททันเบนซิน L200 ดังนั้นโปรดตรวจสอบประเภทของซีลที่ต้องการใช้งานให้ถูกต้อง.


In [43]:
print(response.usage_metadata)

cache_tokens_details=None cached_content_token_count=None candidates_token_count=222 candidates_tokens_details=None prompt_token_count=276 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=276
)] thoughts_token_count=725 tool_use_prompt_token_count=447 tool_use_prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=447
)] total_token_count=1670 traffic_type=None


In [45]:
docs = list(client.file_search_stores.documents.list(parent=store.name))

for d in docs:
    
    print(d.name)
    print(d.display_name)

    # client.file_search_stores.documents.delete(
    #     name=d.name,
    #     config={"force": True},
    # )   
    
    print("-----")

fileSearchStores/kcwkbstore-y3a6az4kie04/documents/vehiclepartsandbearingsrefm-4or40qlqu0q8
Vehicle_Parts_and_Bearings_Ref.md
-----
fileSearchStores/kcwkbstore-y3a6az4kie04/documents/aisinclutchtechnicalmasterm-2my0bljfum3r
Aisin_Clutch_Technical_Master.md
-----
fileSearchStores/kcwkbstore-y3a6az4kie04/documents/kubotatractorfullspecsmd-8evukkvhv5hd
Kubota_Tractor_Full_Specs.md
-----
fileSearchStores/kcwkbstore-y3a6az4kie04/documents/partsmeasurementmd-ocmdqwk7pog9
parts_measurement.md
-----
fileSearchStores/kcwkbstore-y3a6az4kie04/documents/kubotabmd-66egw318hsn3
kubota_B.md
-----
fileSearchStores/kcwkbstore-y3a6az4kie04/documents/kubotamu5501md-v4szon9do2ow
kubota_MU5501.md
-----
fileSearchStores/kcwkbstore-y3a6az4kie04/documents/kubotam8540m9540md-qnfpxiwnicy3
kubota_M8540_M9540.md
-----
fileSearchStores/kcwkbstore-y3a6az4kie04/documents/kubotam7040md-ujjizbrac16o
kubota_M7040.md
-----
fileSearchStores/kcwkbstore-y3a6az4kie04/documents/kubotam6040md-euh4cjo12a5v
kubota_M6040.md
----

In [23]:
# client.file_search_stores.documents.delete(
#     name="fileSearchStores/kcwkbstore-y3a6az4kie04/documents/gshared-driveskcwdatakcwana-0qqj8b6x87gx",
#     config={"force": True},
# )

In [25]:
# client.file_search_stores.delete(
#     name=store.name,
#     config={"force": True}
# )

In [27]:
# import pandas as pd
# from pathlib import Path

# excel_file = r"G:\Shared drives\KCW-Data\kcw_analytics\01_raw\knowledge\24_3_26\ความรู้อะไหล่.xlsx"   # change path
# output_dir = Path(r"G:\Shared drives\KCW-Data\kcw_analytics\01_raw\knowledge\24_3_26\csv_output_2")

# output_dir.mkdir(exist_ok=True)

# xls = pd.ExcelFile(excel_file)

# for sheet in xls.sheet_names:
#     df = pd.read_excel(xls, sheet_name=sheet, header=None)
    
#     # clean sheet name for filename
#     safe_name = "".join(c if c.isalnum() else "_" for c in sheet)

#     out_file = output_dir / f"{safe_name}.csv"
#     df.to_csv(out_file, index=False, header=False, encoding="utf-8-sig")

# print("✅ Export completed")